# Loading Data

In [31]:
import pandas as pd
import numpy as np
import os

In [32]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/1


In [33]:
df = pd.read_csv(path + "/ganjoo_ipl_bbb.csv")

# Cleaning Data

In [34]:
df.drop(columns=['wagonX', 'wagonY', 'wagonZone', 'line','length', 'shot', 'control', 'predscore', 'wprob'], inplace=True)

In [35]:
df.columns

Index(['p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl',
       'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal',
       'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs',
       'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
       'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
       'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'date',
       'year', 'ground', 'country', 'winner', 'toss', 'competition',
       'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced',
       'bowlruns', 'bat_out'],
      dtype='object')

# Converting BBB to MBM

In [36]:
import pandas as pd

def process_batting_scorecard_from_kaggle(df):
    if df.empty:
        return pd.DataFrame()

    batting_stats = {}

    for _, ball in df.iterrows():
        match_id = ball['p_match']
        match_date = ball['date'] if 'date' in df.columns else ""
        venue = ball['ground'] if 'ground' in df.columns else ""

        striker = pd.to_numeric(ball['p_bat'], errors='coerce')
        batsman_name = ball['bat']
        bowler_name = ball['bowl']
        batting_team = ball['team_bat']
        bowling_team = ball['team_bowl']

        key = (match_id, batting_team, striker)
        if key not in batting_stats:
            batting_stats[key] = {
                'batsman_name': batsman_name,
                'bowler_name': "",
                'runs': 0,
                'balls': 0,  # Use 'ballfaced' for accurate counting
                'fours': 0,
                'sixes': 0,
                'isOut': False,
                'wicketType': '',
                'innings': ball['inns'],
                'p_bat_id': striker,
                'p_bowl_id': None,
                'bat_team': batting_team,
                'bowl_team': bowling_team,
                'date': match_date,
                'venue': venue
            }
        stats = batting_stats[key]

        runs = ball['batruns']
        stats['runs'] += runs

        # Count boundaries
        if runs == 4:
            stats['fours'] += 1
        elif runs == 6:
            stats['sixes'] += 1

        # Use 'ballfaced' column to count only valid balls faced
        if ball['ballfaced'] == 1:
            stats['balls'] += 1

        # Check if batsman is out
        out_status = str(ball['out']).strip().upper()
        dismissed_player = str(ball['p_out']).strip()
        if out_status == 'TRUE' and dismissed_player == str(striker):
            stats['isOut'] = True
            stats['wicketType'] = str(ball['dismissal']).strip()
            stats['bowler_name'] = bowler_name
            stats['p_bowl_id'] = pd.to_numeric(ball['p_out'], errors='coerce')

        batting_stats[key] = stats

    results = []
    for (match_id, team, player), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'MatchID': match_id,
            'date': stats['date'],
            'venue': stats['venue'],
            'innings': stats['innings'],
            'batsman': player,
            'batsman_name': stats['batsman_name'],
            'bowler_name': stats['bowler_name'],
            'bowl_id': stats['p_bowl_id'],
            'runs': stats['runs'],
            'balls': stats['balls'],  # Now correctly counts only valid balls
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'strikeRate': round(strike_rate, 2),
            'isOut': stats['isOut'],
            'wicketType': stats['wicketType'] if stats['isOut'] else 'not out',
            'bat_team': stats['bat_team'],
            'bowl_team': stats['bowl_team']
        })

    return pd.DataFrame(results)


In [37]:
batting_card=process_batting_scorecard_from_kaggle(df)

In [38]:
batting_card.sort_values(by=['date', 'MatchID', 'innings'], inplace=True)

In [39]:
batting_card = batting_card.rename(columns={'MatchID': 'match_id','innings':'inns','batsman':'batter_id','batsman_name':'batter','bowler_name':'bowler','bowl_id':'bowler_id','strikeRate':'sr'})

In [40]:
batting_card = batting_card.reindex(columns=['date', 'match_id', 'bat_team', 'bowl_team', 'inns','batter_id','batter','runs','balls','fours','sixes','sr','isOut','wicketType','bowler_id', 'bowler','venue'])

In [41]:
batting_card['venue'] = batting_card['venue'].str.split(',').str.get(0)

In [42]:
batting_card.sample()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
1197,2015-04-16,829727,Sunrisers Hyderabad,Rajasthan Royals,1,32102,Naman Ojha,25,32,3,0,78.12,True,bowled,32102.0,Pravin Tambe,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...


In [43]:
batting_card[batting_card['match_id'] == 980923]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
2778,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,33141,Ambati Rayudu,54,49,3,2,110.20,True,caught,33141.0,Barinder Sran,Rajiv Gandhi International Stadium
2860,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,226492,Martin Guptill,2,4,0,0,50.00,True,caught,226492.0,Bhuvneshwar Kumar,Rajiv Gandhi International Stadium
2986,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,32242,Parthiv Patel,10,12,2,0,83.33,True,bowled,32242.0,Barinder Sran,Rajiv Gandhi International Stadium
2997,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,471342,Krunal Pandya,49,28,3,3,175.00,False,not out,NaN,,Rajiv Gandhi International Stadium
3064,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,29264,Harbhajan Singh,1,3,0,0,33.33,False,not out,NaN,,Rajiv Gandhi International Stadium
6203,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,308967,Jos Buttler,11,11,1,0,100.00,True,caught,308967.0,Barinder Sran,Rajiv Gandhi International Stadium
6206,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,34102,Rohit Sharma,5,8,0,0,62.50,True,run out,34102.0,Bipul Sharma,Rajiv Gandhi International Stadium
6726,2016-04-18,980923,Mumbai Indians,Sunrisers Hyderabad,1,625371,Hardik Pandya,2,5,0,0,40.00,True,bowled,625371.0,Mustafizur Rahman,Rajiv Gandhi International Stadium
1515,2016-04-18,980923,Sunrisers Hyderabad,Mumbai Indians,2,5961,Moises Henriques,20,22,1,1,90.91,True,caught,5961.0,Tim Southee,Rajiv Gandhi International Stadium
1563,2016-04-18,980923,Sunrisers Hyderabad,Mumbai Indians,2,24598,Eoin Morgan,11,12,1,0,91.67,True,caught,24598.0,Tim Southee,Rajiv Gandhi International Stadium


In [44]:
batting_card.to_csv('ganjoo_ipl_mbm_bat.csv', index=False)